In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import xgboost as xgb
from bayes_opt import BayesianOptimization

np.random.seed(42)




In [2]:
usecols_train = [
    "fare_amount",
    "pickup_datetime",
    "pickup_longitude",
    "pickup_latitude",
    "dropoff_longitude",
    "dropoff_latitude",
    "passenger_count",
]
dtype_map = {
    "fare_amount": "float32",
    "pickup_longitude": "float32",
    "pickup_latitude": "float32",
    "dropoff_longitude": "float32",
    "dropoff_latitude": "float32",
    "passenger_count": "int8",
}
train_path = Path("../input/train.csv")

chunks = pd.read_csv(
    train_path,
    usecols=usecols_train,
    parse_dates=["pickup_datetime"],
    dtype=dtype_map,
    low_memory=False,
    chunksize=5_000_000,  # increased from 2 M to speed up reading
)

filtered_chunks = []
target_rows = 4_000_000  # collect more rows for better training
for chunk in chunks:
    mask = (chunk["fare_amount"] > 0.05) & (chunk["passenger_count"] > 0)
    mask &= chunk["pickup_longitude"].between(-75, -73)
    mask &= chunk["dropoff_longitude"].between(-75, -73)
    mask &= chunk["pickup_latitude"].between(40, 42)
    mask &= chunk["dropoff_latitude"].between(40, 42)
    mask &= chunk["passenger_count"].between(0, 8)
    mask &= chunk["fare_amount"].between(0, 250)
    filtered = chunk.loc[mask].sample(frac=0.5, random_state=42)
    filtered_chunks.append(filtered)
    if sum(len(c) for c in filtered_chunks) >= target_rows:
        break

df_train = pd.concat(filtered_chunks, ignore_index=True)




In [3]:
usecols_test = [
    "key",
    "pickup_datetime",
    "pickup_longitude",
    "pickup_latitude",
    "dropoff_longitude",
    "dropoff_latitude",
    "passenger_count",
]
test_path = Path("../input/test.csv")
df_test = pd.read_csv(
    test_path,
    usecols=usecols_test,
    parse_dates=["pickup_datetime"],
    low_memory=True,
)




In [4]:
def haversine_miles(lat1, lon1, lat2, lon2):
    """Vectorised haversine distance in miles."""
    p = np.pi / 180.0
    a = (
        0.5
        - np.cos((lat2 - lat1) * p) / 2
        + np.cos(lat1 * p) * np.cos(lat2 * p) * (1 - np.cos((lon2 - lon1) * p)) / 2
    )
    return 0.6213712 * 12742 * np.arcsin(np.sqrt(a))


df_train["distance_miles"] = haversine_miles(
    df_train["pickup_latitude"],
    df_train["pickup_longitude"],
    df_train["dropoff_latitude"],
    df_train["dropoff_longitude"],
).astype(np.float32)
df_train["distance_squared"] = (df_train["distance_miles"] ** 2).astype(np.float32)
df_train["log_distance"] = np.log1p(df_train["distance_miles"]).astype(np.float32)

df_test["distance_miles"] = haversine_miles(
    df_test["pickup_latitude"],
    df_test["pickup_longitude"],
    df_test["dropoff_latitude"],
    df_test["dropoff_longitude"],
).astype(np.float32)
df_test["distance_squared"] = (df_test["distance_miles"] ** 2).astype(np.float32)
df_test["log_distance"] = np.log1p(df_test["distance_miles"]).astype(np.float32)

for df in (df_train, df_test):
    df["year"] = df["pickup_datetime"].dt.year.astype(np.int16)
    df["hour"] = df["pickup_datetime"].dt.hour.astype(np.int8)
    df["month"] = df["pickup_datetime"].dt.month.astype(np.int8)
    df["dayofweek"] = df["pickup_datetime"].dt.dayofweek.astype(np.int8)




In [5]:
features = [
    "year",
    "hour",
    "month",
    "dayofweek",
    "distance_miles",
    "distance_squared",
    "log_distance",
    "passenger_count",
    "pickup_longitude",
    "pickup_latitude",
    "dropoff_longitude",
    "dropoff_latitude",
]
X = df_train[features]
y = df_train["fare_amount"]
X_test = df_test[features]

subset_frac = 0.30
X_sub, _, y_sub, _ = train_test_split(
    X, y, train_size=subset_frac, random_state=42, stratify=None
)
X_train_bo, X_val_bo, y_train_bo, y_val_bo = train_test_split(
    X_sub, y_sub, test_size=0.2, random_state=42
)




In [6]:
dtrain_bo = xgb.DMatrix(X_train_bo.astype(np.float32), label=y_train_bo)
dval_bo = xgb.DMatrix(X_val_bo.astype(np.float32), label=y_val_bo)


def xgb_eva(max_depth, gamma, colsample_bytree):
    params = {
        "eval_metric": "rmse",
        "objective": "reg:squarederror",
        "max_depth": int(max_depth),
        "subsample": 0.8,
        "eta": 0.1,
        "gamma": gamma,
        "colsample_bytree": colsample_bytree,
        "tree_method": "hist",
        "max_bin": 64,
        "nthread": 4,
    }
    model = xgb.train(
        params,
        dtrain_bo,
        num_boost_round=400,  # more rounds to let BO explore
        evals=[(dval_bo, "eval")],
        early_stopping_rounds=30,  # a little more patience
        verbose_eval=False,
    )
    return -float(model.best_score)




In [7]:
xgb_bo = BayesianOptimization(
    f=xgb_eva,
    pbounds={
        "max_depth": (3, 8),
        "gamma": (0.0, 1.0),
        "colsample_bytree": (0.3, 0.9),
    },
    random_state=42,
    verbose=0,
)
xgb_bo.maximize(init_points=2, n_iter=5)  # a couple more BO iterations




In [8]:
best_params = xgb_bo.max["params"]
best_params["max_depth"] = int(best_params["max_depth"])
best_params["objective"] = "reg:squarederror"
best_params["eval_metric"] = "rmse"
best_params["subsample"] = 0.8
best_params["eta"] = 0.05  # lower learning rate for better generalisation
best_params["tree_method"] = "hist"
best_params["max_bin"] = 64
best_params["nthread"] = 4




In [9]:
# Train final model on the original fare values (no log transform) because the competition metric evaluates RMSE directly.
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

dtrain = xgb.DMatrix(X_train.astype(np.float32), label=y_train)
dval = xgb.DMatrix(X_val.astype(np.float32), label=y_val)

model = xgb.train(
    best_params,
    dtrain,
    num_boost_round=2000,
    evals=[(dval, "eval")],
    early_stopping_rounds=50,
    verbose_eval=False,
)




In [10]:
# Predict on the test set using the model trained on the original scale.
X_test_dm = xgb.DMatrix(X_test.astype(np.float32))
y_pred = model.predict(X_test_dm)

submission = pd.DataFrame({"key": df_test["key"], "fare_amount": y_pred})
submission.to_csv("submission.csv", index=False)
print("Submission file written to submission.csv")

Submission file written to submission.csv
